# Random Forest Nuclei Classification

Train a RF nuclei classifier to distinguish hyphal nuclei, epithelial nuclei, and invaded epithelial nuclei

## Load 3D stack data

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern, load_tiff_stack
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = ("~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI",)
file_pattern = ("CROP_*",
                "MAX_*",
                "SHARPEST*",
                "C[0-9]*",
                )

config = {
    "preprocessing": {
        "resize_image": True,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit"
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[0], file_pattern[3], verbose=True)

# Create output directory
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/rf_nuclei")

# Assuming 'files' is defined in your notebook environment as a list of Path objects
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

print(f"The current output dir is {current_output_dir}")

In [ ]:
from image_processing_tools.util.load_files import load_tiff_stack
from image_processing_tools.rf_nuclei.rf_pre_processing import remove_outliers

data = load_tiff_stack(files[1])
# data = remove_outliers(data)

## Detect in-focus slices

In [ ]:
from image_processing_tools.rf_nuclei.rf_pre_processing import plot_z_stack_variance
import numpy as np

focused, unfocused = plot_z_stack_variance(data)
print(np.max(unfocused))
data_focused = data[focused,...]

## Load Background vs. Nuclei model

In [ ]:
from image_processing_tools.rf_nuclei.rf_load_models import load_model
from pathlib import Path

model_path = Path("./rf_models/nuclei_background.joblib")
model = load_model(model_path)

### Test model: does it work across slices

Trained a model on the max projection image. Now we evaluate it across the individual slices to assess its prediction capabilities.

In [ ]:
from image_processing_tools.rf_nuclei.rf_nuclei_bg_prediction import predict_z_stack_and_plot
from image_processing_tools.rf_nuclei.rf_pre_processing import rescale_and_convert

output_filename = current_output_dir/f"{files[1].stem}_nuclei_vs_background.png"

data_test = rescale_and_convert(data_focused)
predict_z_stack_and_plot(data_test, model, output_filename)

## Test filter sets

Need x filters:
- Output 1: for nuclei object detection
- Output 2: Frangi and feature extraction
- Output 3: Sobel and feature extraction
- Output(s) 4: Gabor and feature extraction

In [ ]:
from image_processing_tools.image_class.image_filters import ImageJProcessor
slice = data_focused[10,...]
pipeline_seg = ImageJProcessor(slice)
pipeline_seg.reset()
pipeline_seg.enhance_contrast_clahe(slope=6)
pipeline_seg.gaussian_blur(sigma=3)
# pipeline_seg.fft_bandpass_filter(40,5)
# pipeline_seg.imagej_rolling_ball(radius=200, use_paraboloid=False)
# pipeline_seg.enhance_contrast_clahe(slope=1.5)
filtered_slice = pipeline_seg.return_image()

In [ ]:
from skimage.filters import threshold_otsu
from scipy import ndimage
import matplotlib.pyplot as plt

thresh_val = threshold_otsu(filtered_slice)
binary_mask = filtered_slice > thresh_val
binary_mask_filled = ndimage.binary_fill_holes(binary_mask)

fig, axes = plt.subplots(1,3,figsize=(12,4))
axes[0].imshow(slice, cmap='gray'); axes[0].set_title('Original')
axes[1].imshow(filtered_slice, cmap='gray'); axes[1].set_title('Filtered')
axes[2].imshow(binary_mask_filled, cmap='gray'); axes[2].set_title('Threshold')

for ax in axes: ax.axis('off')
# plt.axis('off')
plt.tight_layout()
plt.show()